# 01 · SSA acquisition and data-quality audit

**Phase 1 — understand the sources before choosing cleaning rules.**

This notebook identifies the national and state releases and audits their published rows. It does not perform EDA, fit models, fill suppressed observations, or declare the entire preparation phase complete.

Read each explanation before running the next cell. The companion [data dictionary](../docs/data_dictionary.md) defines the observation units and fields; [course requirements](../docs/project_requirements.md) records the assessed deliverables.

**Run status, 9 November 2026:** automated requests to both official ZIP URLs returned HTTP 403 in the preparation session. No real dataset audit has been completed or reported. The code supports manual downloads and any source files already present on your machine. Test-fixture checks are described in [the audit status](../docs/data_audit.md).

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from urllib.request import urlopen
from urllib.error import HTTPError, URLError
from zipfile import ZipFile, is_zipfile
import hashlib
import json
import re
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "requirements.txt").is_file(), "Start Jupyter from the repository root."
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ARCHIVE_DIR = RAW_DIR / "archives"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "audit"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "national": "https://www.ssa.gov/oact/babynames/names.zip",
    "state": "https://www.ssa.gov/oact/babynames/state/namesbystate.zip",
}
EXPECTED_STATES = set("AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY".split())
TARGET_END_YEAR = 2025  # A study target, not a claim about the files.
DOWNLOAD_MISSING = True  # False: inspect only local ZIPs / existing extracted files.
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

## 1. Obtain the official archives and record provenance

Download the [national archive](https://www.ssa.gov/oact/babynames/names.zip) and [state archive](https://www.ssa.gov/oact/babynames/state/namesbystate.zip). If automated access is refused, use those links in your browser and place `names.zip` and `namesbystate.zip` in `data/raw/archives/`. Keep the original filenames and record the actual download date separately if it is not known to the notebook.

We cache archives, compute SHA-256 checksums and preserve the archive READMEs. A checksum identifies bytes; it does not certify a release's provenance or completeness. A cached archive is reused, not silently refreshed. Existing raw files are never replaced with different content: investigate release differences first.

An existing extracted dataset can still be inspected without its ZIP, but its archive identity and original download date remain unverified.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def obtain_archive(kind, url):
    """Reuse a local ZIP, or download it; retain the true acquisition metadata."""
    archive_path = ARCHIVE_DIR / url.rsplit("/", 1)[-1]
    metadata_path = archive_path.with_suffix(".provenance.json")
    downloaded = False
    if not archive_path.exists():
        if not DOWNLOAD_MISSING:
            print(f"{kind}: no ZIP; automatic download disabled.")
            return None
        partial = archive_path.with_suffix(".part")
        try:
            with urlopen(url, timeout=60) as response, partial.open("wb") as handle:
                for block in iter(lambda: response.read(1024 * 1024), b""):
                    handle.write(block)
            if not is_zipfile(partial):
                raise ValueError(f"Downloaded response is not a ZIP: {url}")
            partial.rename(archive_path)
            downloaded = True
        except (HTTPError, URLError, TimeoutError) as error:
            print(f"{kind}: download unavailable ({error}). Use the browser link above.")
            return None
        finally:
            partial.unlink(missing_ok=True)
    if not is_zipfile(archive_path):
        raise ValueError(f"Not a valid ZIP: {archive_path.name}")
    checksum = sha256_file(archive_path)
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        if metadata["sha256"] != checksum:
            raise ValueError(f"Archive differs from its recorded checksum: {archive_path.name}")
    else:
        metadata = {
            "source_url": url,
            "acquisition": "downloaded by this notebook" if downloaded else "pre-existing ZIP; provenance needs confirmation",
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat() if downloaded else None,
            "first_identified_at_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": checksum,
            "bytes": archive_path.stat().st_size,
        }
        metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
    return archive_path, metadata


def extract_without_replacement(archive_path, destination):
    """Validate ZIP paths and collisions before writing any new raw files."""
    with ZipFile(archive_path) as archive:
        members = [entry for entry in archive.infolist() if not entry.is_dir()]
        names = [entry.filename for entry in members]
        if len(names) != len(set(names)):
            raise ValueError("ZIP contains duplicate member paths.")
        for entry in members:
            relative = PurePosixPath(entry.filename)
            if relative.is_absolute() or ".." in relative.parts or "\\" in entry.filename:
                raise ValueError(f"Unexpected archive path: {entry.filename}")
            target = destination.joinpath(*relative.parts)
            if target.exists() and (not target.is_file() or target.read_bytes() != archive.read(entry)):
                raise FileExistsError(f"Raw file differs; inspect before replacing: {target}")
        bad_member = archive.testzip()
        if bad_member:
            raise ValueError(f"ZIP integrity check failed: {bad_member}")
        for entry in members:
            target = destination.joinpath(*PurePosixPath(entry.filename).parts)
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("xb") as handle:
                    handle.write(archive.read(entry))
    return names


provenance = {}
for kind, url in SOURCES.items():
    result = obtain_archive(kind, url)
    if result is not None:
        archive_path, metadata = result
        members = extract_without_replacement(archive_path, RAW_DIR / kind)
        provenance[kind] = {**metadata, "members": members}
        print(f"{kind}: verified ZIP; {len(members)} members (including documentation).")
    else:
        provenance[kind] = {"source_url": url, "acquisition": "ZIP unavailable; any existing extracted files have unverified archive provenance"}

## 2. Define the observation units and locate the files

- National: `(year, sex, name)` identifies a published count. Files contain `name, sex, count`; the year comes from `yobYYYY.txt`.
- State: `(state, year, sex, name)` identifies a published count. Files contain `state, sex, year, name, count`.

Read the original archive documentation before interpreting the values. The [SSA qualifications](https://www.ssa.gov/oact/babynames/background.html) explain historical undercoverage, spelling handling and publication suppression.

The inventory deliberately reports missing inputs. An empty directory is not a dataset with zero observations. Files absent from the selected archive are flagged as having unverified provenance: combining files from different releases must not silently satisfy the coverage checks.

In [ ]:
def source_files(kind):
    pattern = r"yob[0-9]{4}\.txt" if kind == "national" else r"[A-Z]{2}\.TXT"
    directory = RAW_DIR / kind
    return sorted(path for path in directory.rglob("*") if path.is_file() and re.fullmatch(pattern, path.name))


files_by_kind = {kind: source_files(kind) for kind in SOURCES}
for kind, files in files_by_kind.items():
    print(f"{kind}: {len(files)} source files located" if files else f"{kind}: NOT AUDITED — no source files available")

## 3. Inspect every published row, one file at a time

Reading one whole file at a time limits memory use while allowing duplicate checks across the entire file. We preserve all text with `keep_default_na=False`: a name resembling a pandas missing-value marker must remain a name. Empty fields are checked explicitly.

Numeric conversion below is for validation only. Invalid values are flagged, not silently repaired or dropped. Duplicate counts mean rows beyond the first occurrence of a natural key. Filename/content checks establish whether partitions are disjoint before interpreting within-file uniqueness as dataset-wide uniqueness.

The year target is 2025; older coverage is reported as a limitation rather than silently extended. Counts below five, invalid categories, unexpected whitespace and missing year/sex groups require investigation. A zero missing-field count does not mean that suppressed names or unrecorded births are present.

In [ ]:
def audit_file(path, kind):
    columns = ["name", "sex", "count"] if kind == "national" else ["state", "sex", "year", "name", "count"]
    frame = pd.read_csv(path, header=None, dtype=str, keep_default_na=False, skip_blank_lines=False, encoding="utf-8", on_bad_lines="error")
    if frame.shape[1] != len(columns):
        raise ValueError(f"Expected {len(columns)} fields, found {frame.shape[1]}")
    frame.columns = columns
    if kind == "national":
        frame["year"] = path.stem[3:]
    empty = frame.isna() | frame.eq("")
    counts = pd.to_numeric(frame["count"], errors="coerce")
    years = pd.to_numeric(frame["year"], errors="coerce")
    invalid_count = counts.isna() | counts.mod(1).ne(0)
    invalid_year = years.isna() | years.mod(1).ne(0) | ~frame["year"].str.fullmatch(r"[0-9]{4}", na=False)
    keys = (["state"] if kind == "state" else []) + ["year", "sex", "name"]
    first_year = 1880 if kind == "national" else 1910
    checks = {
        "empty_fields": int(empty.to_numpy().sum()),
        "invalid_sex_rows": int((~frame["sex"].isin(["F", "M"])).sum()),
        "noninteger_count_rows": int(invalid_count.sum()),
        "counts_below_five": int(counts.lt(5).sum()),
        "invalid_year_rows": int(invalid_year.sum()),
        "years_outside_planned_range": int((years.lt(first_year) | years.gt(TARGET_END_YEAR)).sum()),
        "duplicate_key_rows": int(frame.duplicated(keys).sum()),
        "name_whitespace_rows": int(frame["name"].ne(frame["name"].str.strip()).sum()),
        "short_name_rows": int(frame["name"].str.len().lt(2).sum()),
    }
    if kind == "state":
        checks["invalid_state_rows"] = int((~frame["state"].isin(EXPECTED_STATES)).sum())
        checks["filename_state_mismatch_rows"] = int(frame["state"].ne(path.stem).sum())
    observed_years = sorted(int(value) for value in years[~invalid_year].unique())
    missing_years = sorted(set(range(min(observed_years), max(observed_years) + 1)) - set(observed_years)) if observed_years else []
    observed_groups = set(zip(years[~invalid_year].astype(int), frame.loc[~invalid_year, "sex"]))
    missing_groups = [(year, sex) for year in observed_years for sex in ["F", "M"] if (year, sex) not in observed_groups]
    return {
        "file": path.relative_to(PROJECT_ROOT).as_posix(),
        "sha256": sha256_file(path),
        "rows": len(frame),
        "analytical_columns": len(frame.columns),
        "min_year": min(observed_years) if observed_years else None,
        "max_year": max(observed_years) if observed_years else None,
        "min_count": int(counts[~invalid_count].min()) if (~invalid_count).any() else None,
        "max_count": int(counts[~invalid_count].max()) if (~invalid_count).any() else None,
        "sex_values": sorted(frame["sex"].unique().tolist()),
        "empty_fields_by_column": {column: int(value) for column, value in empty.sum().items()},
        "missing_internal_years": missing_years,
        "missing_year_sex_groups": missing_groups,
        "checks": checks,
    }


def audit_dataset(kind, files, archive_members=None):
    if not files:
        return {"status": "not_audited_no_files", "files": [], "errors": [], "coverage": None, "rows_audited": None}
    results, errors = [], []
    for path in files:
        try:
            results.append(audit_file(path, kind))
        except (ValueError, pd.errors.ParserError, pd.errors.EmptyDataError, UnicodeError, OSError) as error:
            errors.append({"file": path.relative_to(PROJECT_ROOT).as_posix(), "error": str(error)})
    partitions = [path.stem[3:] if kind == "national" else path.stem for path in files]
    repeated_partitions = sorted(partition for partition in set(partitions) if partitions.count(partition) > 1)
    valid_results = [row for row in results if row["min_year"] is not None]
    first_year = 1880 if kind == "national" else 1910
    observed_end = max((row["max_year"] for row in valid_results), default=None)
    missing_partitions = sorted(set(str(year) for year in range(1880, TARGET_END_YEAR + 1)) - set(partitions)) if kind == "national" else sorted(EXPECTED_STATES - set(partitions))
    coverage = {"expected_start": first_year, "target_end": TARGET_END_YEAR, "observed_start": min((row["min_year"] for row in valid_results), default=None), "observed_end": observed_end, "missing_expected_partitions": missing_partitions, "repeated_partitions": repeated_partitions}
    coverage["files_not_covering_target"] = [row["file"] for row in valid_results if kind == "state" and (row["min_year"] != first_year or row["max_year"] != TARGET_END_YEAR)]
    unmapped_files = [path.relative_to(PROJECT_ROOT).as_posix() for path in files if archive_members is None or path.relative_to(RAW_DIR / kind).as_posix() not in archive_members]
    archive_membership = {"status": "unknown" if archive_members is None else "checked", "files_with_unverified_archive_provenance": unmapped_files}
    issues = bool(errors or repeated_partitions or missing_partitions or coverage["files_not_covering_target"] or unmapped_files)
    issues = issues or any(any(row["checks"].values()) or row["missing_internal_years"] or row["missing_year_sex_groups"] for row in results)
    return {"status": "review_required" if issues else "checks_passed_for_published_rows", "rows_audited": sum(row["rows"] for row in results), "files": results, "errors": errors, "coverage": coverage, "archive_membership": archive_membership}


audits = {kind: audit_dataset(kind, files, provenance[kind].get("members")) for kind, files in files_by_kind.items()}
for kind, audit in audits.items():
    print(kind, audit["status"], "| rows audited:", audit["rows_audited"])
    if audit["coverage"]:
        print(json.dumps(audit["coverage"], indent=2))
        print(json.dumps(audit["archive_membership"], indent=2))
    for error in audit["errors"]:
        print("FILE ERROR:", error)

## 4. Save evidence and interpret the checks

The JSON preserves each check and file checksum; the CSV is a compact per-file view. Each run gets its own UTC timestamp folder under `data/processed/audit/`, ignored by Git, so earlier evidence cannot be mistaken for new output. Share a reviewed summary in `docs/data_audit.md` only after a real run, including any unresolved issues. If no files were inspected, this cell writes no audit results; any existing output folders belong to earlier runs.

Passing these checks establishes consistency of the published rows, not complete population coverage. National and state totals may differ because suppression is applied at different geographical levels. Do not fill absent names with zero or automatically calculate female shares of 0 or 1 from a missing opposite-sex row.

In [ ]:
if any(audit["files"] or audit["errors"] for audit in audits.values()):
    run_dir = OUTPUT_DIR / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_dir.mkdir(parents=True, exist_ok=False)
    report = {
        "audited_at_utc": datetime.now(timezone.utc).isoformat(),
        "python_version": sys.version.split()[0],
        "pandas_version": pd.__version__,
        "target_end_year": TARGET_END_YEAR,
        "provenance": provenance,
        "datasets": audits,
    }
    (run_dir / "ssa_audit.json").write_text(json.dumps(report, indent=2) + "\n")
    rows = [{"dataset": kind, **{key: row[key] for key in ["file", "sha256", "rows", "analytical_columns", "min_year", "max_year", "min_count", "max_count"]}, **row["checks"]} for kind, audit in audits.items() for row in audit["files"]]
    if rows:
        pd.DataFrame(rows).to_csv(run_dir / "ssa_file_audit.csv", index=False)
    else:
        print("No file parsed successfully: this run has JSON errors and no per-file CSV.")
    print("This run's audit evidence:", run_dir.relative_to(PROJECT_ROOT))
else:
    print("No datasets were readable. No audit results were written; acquisition remains incomplete.")
    if OUTPUT_DIR.exists():
        print("Existing output folders belong to earlier runs; they are not evidence for this attempt.")

## 5. Discuss before proceeding

1. Which releases and years did we actually inspect? Is 2025 present in both sources? What remains unknown about provenance?
2. Are the keys unique and partition checks valid? Explain any missing fields, missing groups or suspicious entries before deciding on a treatment.
3. The sources have four national or five state analytical fields. Which derived variables would help our research questions? Explain their units and denominators; do not pad the schema to reach a recommended column count.
4. Which preparation decisions remain? Specify suppression handling, comparison periods, rank ties and any minimum-count thresholds before creating EDA measures. A Census lookup is needed for regional comparisons.

**Stop here for this task.** No cleaning choices, EDA findings or modelling results have been fabricated. The next preparation step should be agreed after the group reviews the actual audit.